In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    trim,
    row_number,
    regexp_extract,
    when,
    sum,
    max as spark_max,
    min as spark_min,
    expr,
)
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_settlement_exposure_process(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "SettlementExposureId string, SourceSystem string, PortfolioKey string, PortfolioName string,"
        #     "IngestionTimestamp timestamp, BatchId string, SettlementExposureKey string, InitialSettlementKey int,"
        #     "PreviouSettlementKey int, SettlemenKey int, ExposureKey string, AgreementKey string, PartyKey string, CDI string,"
        #     "CaseCode int, DebtAmount decimal(19,4), PrincipalProposedAmount decimal(19,2), ProposedAmount decimal(19,4),"
        #     "RemainAmount decimal(19,4), SettlementAmount decimal(19,4), ExpectedSettlementAmount decimal(19,4),"
        #     "DiscountAmount decimal(19,4), DiscountPercentage decimal(19,4), BalloonAmount decimal(19,2),"
        #     "BucketCode int, Bucket int, SettlementTypeCode string, SettlementType string, SettlementStatusCode string,"
        #     "SettlementStatus string, CreatedDate timestamp, SubmissionDate timestamp, ApprovalDate date,"
        #     "ImplementationDate timestamp, ExpiredDate timestamp, CancelledDate timestamp, RejectedDate date,"
        #     "OverdueDate date, FirstInstalmentDate date, NextInstalmentDate date, PastInstalmentAmount decimal(19,4),"
        #     "PastInstalmentNumber int, NextInstalmentAmount decimal(19,4), NumOfaccounts int, NumOfinstalments int,"
        #     "TotalPaymentAmount decimal(19,4), IsActiveFlag int, Comments string, KeptPercentage decimal(19,4),"
        #     "IsMigratedFlag boolean, MigrationComments string, PoolName string, AddedManuallyFlag boolean,"
        #     "isPrimarySettlementFlag boolean, ParticipationPercentage decimal(19,4), MaturityPercentage decimal(19,4),"
        #     "MutiplyforLCY decimal(19,4), OtherExpense1 decimal(19,4), OtherExpense2 decimal(19,4), OtherExpense3 decimal(19,4),"
        #     "LastUpdatedDate timestamp, CancellationReasonCode int, CancellationReason string, DiasRFCode string,"
        #     "CtrlMovedFlag boolean,ContractNumber string,CaseFrozenInterestAmtLcy decimal(19,4),CaseProvisionAmtLcy decimal(19,4),"
        #     "MvhoCaseFrozenInterestAmt decimal(19,4),MvhoCaseWriteOffAmt  decimal(19,4), MvhoCaseProvisionAmt  decimal(19,4),"
        #     "SettlementLastInstallmentDate  date, CaseWriteOffAmtLcy  decimal(19,4), StartingOutstandingBalance decimal(19,4), StartingDebtAmount decimal(19,4),"
        #     "StartingFrozen decimal(19,4), StartingWriteOffAmount decimal(19,4), StartingCurrentBucket  int, StartingCaseDenounceStatus string,StartingHostAccountStatus string,
        #     "SepsIsDownPayment boolean,SepsInstallmentAmt  decimal(19,4),AgencyCode string,BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.SettlementExposure"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "profits",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "unit_code_profits",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== clc_pay_promises ==================

        clc_pay_promises_extra_filters = None

        if clc_pay_promises_extra_filters is not None:
            clc_pay_promises_extra_filters.append(date_filter)
            clc_pay_promises_final_filters = " AND ".join(clc_pay_promises_extra_filters)
        else:
            clc_pay_promises_final_filters = date_filter

        clc_pay_promises_selected_fields = [
            "case_id",
            "BusinessDate",
            "cust_id",
            "pay_promise_id",
            "PAY_NUMBER",
            "pay_amn",
            "businessdate",
            "pay_dt",
            "PAY_TOLERANCE",
            "CREATE_DATE"
        ]

        clc_pay_promises = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_clc_pay_promises",
            clc_pay_promises_final_filters,
            clc_pay_promises_selected_fields,
        )

        # ================== clc_case ==================

        clc_case_extra_filters = None

        if clc_case_extra_filters is not None:
            clc_case_filters.append(date_filter)
            clc_case_filters = " AND ".join(clc_case_extra_filters)
        else:
            clc_case_final_filters = date_filter

        clc_case_selected_fields = [
            "CASE_ACC_NUMBER",
            "CASE_ACC_SYSTEM",
            "case_id",
            "CREATE_TYPE",
            "CASE_UNIT",
            "CASE_CUSTOMER_ID",
            "BusinessDate",
        ]

        clc_case = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_clc_case",
            clc_case_final_filters,
            clc_case_selected_fields,
        )

        # ================== clc_promises_status ==================

        clc_promises_status_extra_filters = None

        if clc_promises_status_extra_filters is not None:
            clc_promises_status_filters.append(date_filter)
            clc_promises_status_filters = " AND ".join(clc_promises_status_extra_filters)
        else:
            clc_promises_status_final_filters = date_filter

        clc_promises_status_selected_fields = [
            "pay_promise_id",
            "BusinessDate",
            "FK_CLC_SECOND_STS_PREV",
            "FK_CLC_JUST_REASON_PREV",
            "TRX_DATE",
        ]

        clc_promises_status = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_clc_promises_status",
            clc_promises_status_final_filters,
            clc_promises_status_selected_fields,
        )

        # ================== customer_rf_codes ==================

        customer_rf_codes_status_extra_filters = None

        if customer_rf_codes_status_extra_filters is not None:
            customer_rf_codes_filters.append(date_filter)
            customer_rf_codes_filters = " AND ".join(customer_rf_codes_extra_filters)
        else:
            customer_rf_codes_final_filters = date_filter

        customer_rf_codes_selected_fields = [
            "ACCOUNT_NUMBER",
            "ACCOUNT_SYSTEM",
            "CUSTOMER_ID_NO",
            "CUSTOMER_CODE",
            "BusinessDate",
            "CUST_RF_ID",
        ]

        customer_rf_codes = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_customer_rf_codes",
            customer_rf_codes_final_filters,
            customer_rf_codes_selected_fields,
        )

        # ================== Apply row_number() to clc_promises_status ==================
        window_spec = Window.partitionBy(
            "pay_promise_id", "BusinessDate"
        ).orderBy(
            clc_promises_status["TRX_DATE"].desc()
        )

        clc_promises_status_with_rn = clc_promises_status.withColumn(
            "rn", 
            row_number().over(window_spec)
        ).filter(
            "rn = 1"
        ).drop(
            "rn"
        )

        # Derive LastUpdatedDate as the last time the status changed
        current_status_per_promise = clc_promises_status_with_rn.select(
            col("pay_promise_id").alias("_promise_id"),
            col("FK_CLC_SECOND_STS_PREV").alias("_current_status")
        )
        status_change_dates = (
            clc_promises_status
            .join(current_status_per_promise,
                  (clc_promises_status["pay_promise_id"] == col("_promise_id")) &
                  (clc_promises_status["FK_CLC_SECOND_STS_PREV"] == col("_current_status")),
                  "inner")
            .groupBy("_promise_id")
            .agg(F.min("TRX_DATE").alias("LastUpdatedDate"))
            .withColumnRenamed("_promise_id", "pay_promise_id")
        )
        clc_promises_status_with_rn = clc_promises_status_with_rn.join(
            status_change_dates, "pay_promise_id", "left"
        ).withColumnRenamed("pay_promise_id", "_status_promise_id")

        # ================== select distinct from customer_rf_codes ==================
        customer_rf_codes_distinct = customer_rf_codes.select(
            "ACCOUNT_NUMBER", "ACCOUNT_SYSTEM", "CUSTOMER_ID_NO", "CUSTOMER_CODE", "BusinessDate", "CUST_RF_ID"
        ).distinct()
        
        # ================== Join conditions ==================
        clc_case_clc_pay_promises_join_condition = (
            clc_case["case_id"] == clc_pay_promises["case_id"]
        )

        clc_pay_promises_clc_promises_status_join_condition = (
            clc_pay_promises["pay_promise_id"] == clc_promises_status_with_rn["_status_promise_id"]
        )

        clc_case_customer_rf_codes_join_condition = (
            (trim(clc_case["CASE_ACC_NUMBER"]) == customer_rf_codes_distinct["ACCOUNT_NUMBER"]) & 
            (clc_case["CASE_ACC_SYSTEM"].cast("int") == customer_rf_codes_distinct["ACCOUNT_SYSTEM"].cast("int")) &
            (clc_case["CASE_CUSTOMER_ID"] == customer_rf_codes_distinct["CUSTOMER_CODE"].cast("int"))
        )

        clc_case_portfolio_join_condition = (
            clc_case["case_unit"] == portfolio["unit_code_profits"]
        )

        # ================== Joins ==================
        target_table_profits = clc_case.join(
            clc_pay_promises, 
            clc_case_clc_pay_promises_join_condition, 
            "left"
        ).join(
            clc_promises_status_with_rn, 
            clc_pay_promises_clc_promises_status_join_condition, 
            "left"
        ).drop("_status_promise_id").join(
            customer_rf_codes_distinct, 
            clc_case_customer_rf_codes_join_condition, 
            "left"
        ).join(
            portfolio, 
            clc_case_portfolio_join_condition, 
            "left"
        )

        # ================== HashKeys ==================
        # Exposure-level grain (matching QCA pattern: settlement_code;exposure_account)
        target_table_profits_SettlementId_primary_key_fields = [
            clc_pay_promises.pay_promise_id,
            clc_case.CASE_ACC_NUMBER,
            lit("profits"),
        ]

        target_table_profits_SettlementExposureKey_key_fields = [
            clc_pay_promises.pay_promise_id,
            clc_case.CASE_ACC_NUMBER,
        ]

        # ================== Target table (hash keys and calculations) ==================
        # sum of all installment amounts per settlement
        settlement_amount_agg = (
            clc_pay_promises
            .groupBy("pay_promise_id")
            .agg(
                sum("pay_amn").alias("SettlementAmountCalc"),
                spark_max("pay_dt").alias("LastPayDate"),
                sum("PAY_TOLERANCE").alias("TotalPayTolerance"),
                spark_min("CREATE_DATE").alias("CreatedDateCalc")
            )
            .withColumn("ExpiredDateCalc", expr("date_add(LastPayDate, cast(coalesce(TotalPayTolerance, 0) as int))"))
            .withColumnRenamed("pay_promise_id", "_agg_promise_id")
        )
        target_table_profits = target_table_profits.join(
            settlement_amount_agg,
            clc_pay_promises["pay_promise_id"] == settlement_amount_agg["_agg_promise_id"],
            "left"
        ).drop("_agg_promise_id")

        target_table_profits = target_table_profits.withColumn(
            "SettlementExposureId",
            sha2(
                concat_ws(";", *target_table_profits_SettlementId_primary_key_fields),
                256,
            ),
        ).withColumn(
            "SettlementExposureKey",
            concat_ws(";", *target_table_profits_SettlementExposureKey_key_fields),
        ).withColumn(
            "AgreementKey",
            when(
                col("CASE_ACC_SYSTEM") == 19, 
                col("CASE_ACC_NUMBER")
            ).otherwise(None)
        ).withColumn(
            "ExposureKey",
            when(
                col("CASE_ACC_SYSTEM") == 4,
                regexp_extract(col("CASE_ACC_NUMBER"), r'\d.*', 0)
            ).otherwise(None)
        ).withColumn(
            "SettlementStatus",
            when(col("FK_CLC_SECOND_STS_PREV")  == 1, "PENDING FOR IMPLEMENTATION")
            .when(col("FK_CLC_SECOND_STS_PREV") == 2, "CANCELLED-AUTO")
            .when(col("FK_CLC_SECOND_STS_PREV") == 3, "IMPLEMENTED")
            .when(col("FK_CLC_SECOND_STS_PREV") == 4, "FULFILLED")
            .when(col("FK_CLC_SECOND_STS_PREV") == 5, "NOT FULFILLED")
            .when(col("FK_CLC_SECOND_STS_PREV") == 6, "CANCELLED-MANUAL")
            .when(col("FK_CLC_SECOND_STS_PREV") == 7, "LOAN CLOSED")
            .otherwise(None)
        ).withColumn(
            "CancellationReason",
            when(col("FK_CLC_JUST_REASON_PREV")  == 0, "ΝΑ ΣΥΜΠΛΗΡΩΘΕΙ")
            .when(col("FK_CLC_JUST_REASON_PREV") == 1, "ΚΑΤΑΧΩΡΗΣΗ ΝΕΟΥ ΠΛΑΝΟΥ")
            .when(col("FK_CLC_JUST_REASON_PREV") == 2, "ΜΗ ΤΗΡΗΣΗ ΠΡΟΚΑΤΑΒΟΛΗΣ")
            .when(col("FK_CLC_JUST_REASON_PREV") == 3, "ΤΗΡΗΣΗ ΠΟΣΟΥ 1ης ΔΟΣΗΣ")
            .when(col("FK_CLC_JUST_REASON_PREV") == 4, "ΟΛΟΚΛΗΡΩΣΗ ΠΛΗΡΩΜΩΝ")
            .when(col("FK_CLC_JUST_REASON_PREV") == 5, "ΜΗ ΤΗΡΗΣΗ ΠΛΗΡΩΜΗΣ")
            .when(col("FK_CLC_JUST_REASON_PREV") == 6, "ΑΠΟΦΑΣΗ ΠΕΛΑΤΗ")
            .when(col("FK_CLC_JUST_REASON_PREV") == 7, "ΑΠΟΠΛΗΡΩΜΗ ΔΑΝΕΙΟΥ")
            .when(col("FK_CLC_JUST_REASON_PREV") == 8, "ΤΗΡΗΣΗ ΠΛΗΡΩΜΩΝ")
            .otherwise(None)
        ).withColumn(
            "SettlementType",
            when(col("CREATE_TYPE").cast("int") == 0, "ΝΑ ΣΥΜΠΛΗΡΩΘΕΙ")
            .when(col("CREATE_TYPE").cast("int") == 1, "ΔΙΑΓΡΑΦΗ ΥΠΟΛΟΙΠΟΥ")
            .when(col("CREATE_TYPE").cast("int") == 2, "ΑΡΣΗ ΕΓΓΥΗΣΗΣ")
            .when(col("CREATE_TYPE").cast("int") == 3, "ΠΡΟΣΩΡΙΝΗ ΑΠΟΦ Ν3869")
            .when(col("CREATE_TYPE").cast("int") == 4, "ΑΛΛΟΣ")
            .when(col("CREATE_TYPE").cast("int") == 5, "ΟΡΙΣΤΙΚΗ ΑΠΟΦ Ν3869")
            .otherwise(None)
        )

        # ================== Selection of Final Columns ==================

        result_df_profits = target_table_profits.select(
            target_table_profits.SettlementExposureId.alias("SettlementExposureId"),
            lit("profits").cast("string").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("string").alias("BatchId"),
            target_table_profits.SettlementExposureKey.alias("SettlementExposureKey"),
            lit(None).cast("int").alias(
                "InitialSettlementKey"
            ),
            lit(None).cast("int").alias("PreviouSettlementKey"),
            clc_pay_promises.pay_promise_id.cast("int").alias("SettlementKey"),
            trim(target_table_profits.ExposureKey).cast("string").alias("ExposureKey"),
            trim(target_table_profits.AgreementKey).cast("string").alias("AgreementKey"),
            target_table_profits.CASE_CUSTOMER_ID.cast("string").alias("PartyKey"),
            customer_rf_codes_distinct.CUSTOMER_ID_NO.cast("string").alias("CDI"),
            lit(None).cast("int").alias("CaseCode"),
            lit(None).cast("decimal(19,4)").alias("DebtAmount"),
            lit(None).cast("decimal(19,2)").alias(
                "PrincipalProposedAmount"
            ),
            lit(None).cast("decimal(19,4)").alias("ProposedAmount"),
            lit(None).cast("decimal(19,4)").alias("RemainAmount"),
            target_table_profits.SettlementAmountCalc.cast("decimal(19,4)").alias(
                "SettlementAmount"
            ),
            lit(None).cast("decimal(19,4)").alias(
                "ExpectedSettlementAmount"
            ),
            lit(None).cast("decimal(19,4)").alias("DiscountAmount"),
            lit(None).cast("decimal(19,4)").alias("DiscountPercentage"),
            lit(None).cast("decimal(19,2)").alias("BalloonAmount"),
            lit(None).cast("int").alias("BucketCode"),
            lit(None).cast("int").alias("Bucket"),
            target_table_profits.CREATE_TYPE.cast("string").alias("SettlementTypeCode"),
            target_table_profits.SettlementType.cast("string").alias("SettlementType"),
            target_table_profits.FK_CLC_SECOND_STS_PREV.cast("string").alias("SettlementStatusCode"),
            target_table_profits.SettlementStatus.cast("string").alias("SettlementStatus"),
            target_table_profits.CreatedDateCalc.cast("timestamp").alias("CreatedDate"),
            lit(None).cast("timestamp").alias("SubmissionDate"),
            lit(None).cast("date").alias("ApprovalDate"),
            lit(None).cast("timestamp").alias("ImplementationDate"),
            target_table_profits.ExpiredDateCalc.cast("timestamp").alias("ExpiredDate"),
            lit(None).cast("timestamp").alias("CancelledDate"),
            lit(None).cast("date").alias("RejectedDate"),
            lit(None).cast("date").alias("OverdueDate"),
            lit(None).cast("date").alias("FirstInstalmentDate"),
            lit(None).cast("date").alias("NextInstalmentDate"),
            lit(None).cast("decimal(19,4)").alias("PastInstalmentAmount"),
            lit(None).cast("int").alias("PastInstalmentNumber"),
            lit(None).cast("decimal(19,4)").alias(
                "NextInstalmentAmount"
            ),
            lit(None).cast("int").alias("NumOfaccounts"),
            lit(None).cast("int").alias("NumOfinstalments"),
            lit(None).cast("decimal(19,4)").alias(
                "TotalPaymentAmount"
            ),
            when(col("FK_CLC_SECOND_STS_PREV").isin(1, 3, 4, 5, 7), lit(1)).otherwise(lit(0)).cast("int").alias("IsActiveFlag"),
            lit(None).cast("string").alias("Comments"),
            lit(None).cast("decimal(19,4)").alias("KeptPercentage"),
            lit(None).cast("boolean").alias("IsMigratedFlag"),
            lit(None).cast("string").alias("MigrationComments"),
            lit(None).cast("string").alias("PoolName"),
            lit(None).cast("boolean").alias(
                "AddedManuallyFlag"
            ),
            lit(True).cast("boolean").alias(
                "isPrimarySettlementFlag"
            ),
            lit(None).cast("decimal(19,4)").alias(
                "ParticipationPercentage"
            ),
            lit(None).cast("decimal(19,4)").alias("MaturityPercentage"),
            lit(None).cast("decimal(19,4)").alias("MutiplyforLCY"),
            lit(None).cast("decimal(19,4)").alias("OtherExpense1"),
            lit(None).cast("decimal(19,4)").alias("OtherExpense2"),
            lit(None).cast("decimal(19,4)").alias("OtherExpense3"),
            clc_promises_status_with_rn["LastUpdatedDate"].cast("timestamp").alias("LastUpdatedDate"),
            target_table_profits.FK_CLC_JUST_REASON_PREV.cast("int").alias("CancellationReasonCode"),
            target_table_profits.CancellationReason.cast("string").alias("CancellationReason"),
            target_table_profits.CUST_RF_ID.cast("string").alias("DiasRFCode"),
            clc_case.BusinessDate.cast("date").alias("BusinessDate")
            ,lit(None).cast("decimal(19,4)").alias("MvhoCaseFrozenInterestAmt")
            ,lit(None).cast("decimal(19,4)").alias("MvhoCaseWriteOffAmt")
            ,lit(None).cast("decimal(19,4)").alias("MvhoCaseProvisionAmt")
            ,lit(None).cast("date").alias("SettlementLastInstallmentDate")
            ,lit(None).cast("decimal(19,4)").alias("CaseWriteOffAmtLcy")
            ,lit(None).cast("decimal(19,4)").alias("StartingOutstandingBalance")
            ,lit(None).cast("decimal(19,4)").alias("StartingDebtAmount")
            ,lit(None).cast("decimal(19,4)").alias("StartingFrozen")
            ,lit(None).cast("decimal(19,4)").alias("StartingWriteOffAmount")
            ,lit(None).cast("int").alias("StartingCurrentBucket")
            ,lit(None).cast("string").alias("StartingCaseDenounceStatus")
            ,lit(None).cast("string").alias("StartingHostAccountStatus")
            ,lit(None).cast("boolean").alias("SepsIsDownPayment")
            ,lit(None).cast("decimal(19,4)").alias("SepsInstallmentAmt")
            ,lit(None).cast("string").alias("AgencyCode")
            ,lit(None).cast("boolean").alias("CtrlMovedFlag")
            ,lit(None).cast("string").alias("ContractNumber")
            ,lit(None).cast("decimal(19,4)").alias("CaseFrozenInterestAmtLcy")
            ,lit(None).cast("decimal(19,4)").alias("CaseProvisionAmtLcy")
            ,lit(None).cast("int").alias("DaysPastDue")
        )

        # ================== Dedup to Exposure Level ==================
        # Profits source has 1 row per (SettlementKey, ExposureKey, PAY_NUMBER).
        # All non-key fields are identical across PAY_NUMBER rows.
        # Dedup to 1 row per (SettlementKey, ExposureKey/CDI) to match QCA grain.
        w = Window.partitionBy(
            "SettlementKey", coalesce("ExposureKey", "CDI")
        ).orderBy(lit(1))
        result_df_profits = (
            result_df_profits
            .withColumn("_rn", row_number().over(w))
            .filter("_rn = 1")
            .drop("_rn")
        )

        result_df_profits.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_settlement_exposure_process(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)